<a href="https://colab.research.google.com/github/yashicam22/Stock-Scorecard/blob/main/Stock_Scorecard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/stocks_13_labeled.csv', index_col='Date', parse_dates=True)
df.head()

,Ticker,Open,High,Low,Close,Volume,Label,Year
Date,,,,,,,,
2021-09-01,AAPL,152.830002,154.979996,152.339996,152.509995,80313700,0.0,2021
2021-09-02,AAPL,153.869995,154.720001,152.399994,153.649994,71115500,0.0,2021
2021-09-03,AAPL,153.759995,154.630005,153.089996,154.300003,57808700,0.0,2021
2021-09-07,AAPL,154.970001,157.259995,154.389999,156.690002,82278300,0.0,2021
2021-09-08,AAPL,156.979996,157.039993,153.979996,155.110001,74420200,0.0,2021


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/stocks_13_labeled.csv', index_col='Date', parse_dates=True)
df = df.sort_index()

VAULT = ['GS', 'UNH', 'CVX', 'HD', 'NFLX']
df = df[~df['Ticker'].isin(VAULT)].copy()   # vault stocks removed

print(df.shape)
print(df['Ticker'].unique())

(16302, 8)
['AAPL' 'TSLA' 'XOM' 'AMD' 'AVGO' 'NVDA' 'AMZN' 'MSFT' 'COST' 'GOOGL'
 'META' 'LLY' 'JPM']


In [ ]:
df['SMA20'] = df.groupby('Ticker')['Close'].transform(lambda s: s.rolling(20).mean())
df['SMA50'] = df.groupby('Ticker')['Close'].transform(lambda s: s.rolling(50).mean())
df['EMA20'] = df.groupby('Ticker')['Close'].transform(lambda s: s.ewm(span=20, adjust=False).mean())

df['TP']  = (df['High'] + df['Low'] + df['Close']) / 3
df['TPV'] = df['TP'] * df['Volume']
df['VWAP20'] = (
    df.groupby('Ticker')['TPV'].transform(lambda s: s.rolling(20).sum())
    / df.groupby('Ticker')['Volume'].transform(lambda s: s.rolling(20).sum())
)

df['Return'] = df.groupby('Ticker')['Close'].pct_change()

In [ ]:
df = df.reset_index()

def compute_macd(close):
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    return macd, signal

macd_list, signal_list = [], []
for ticker, g in df.groupby('Ticker'):
    macd, signal = compute_macd(g['Close'])
    macd_list.append(macd)
    signal_list.append(signal)
df['MACD'] = pd.concat(macd_list)
df['Signal'] = pd.concat(signal_list)

def compute_rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

rsi_list = []
for ticker, g in df.groupby('Ticker'):
    rsi_list.append(compute_rsi(g['Close']))
df['RSI14'] = pd.concat(rsi_list)

df = df.set_index('Date').sort_index()
df = df.drop(columns=['TP', 'TPV'], errors='ignore')

print(df[['SMA20','SMA50','EMA20','VWAP20','MACD','Signal','RSI14']].describe())

              SMA20         SMA50         EMA20        VWAP20          MACD  \
count  16055.000000  15665.000000  16302.000000  16055.000000  16302.000000   
mean     286.460905    285.443441    285.039229    286.113202      1.523810   
std      233.375545    232.034070    232.230935    233.017767      8.177218   
min       12.216950     13.012820     12.363816     12.202324    -38.152191   
25%      129.718250    128.894800    129.544777    129.136203     -1.903681   
50%      202.725500    203.353400    202.951499    202.609364      0.837665   
75%      356.993246    354.453799    354.476860    356.291946      4.234634   
max     1204.303003   1188.004004   1213.259921   1201.767118     65.717355   

             Signal         RSI14  
count  16302.000000  16120.000000  
mean       1.520002     53.335299  
std        7.649061     17.253829  
min      -33.932267      3.717473  
25%       -1.704804     40.660531  
50%        0.800965     53.155809  
75%        4.064240     65.918526  


In [ ]:
df['ret_5d']   = df.groupby('Ticker')['Close'].pct_change(5)
df['ret_20d']  = df.groupby('Ticker')['Close'].pct_change(20)
df['px_sma20'] = df['Close'] / df['SMA20'] - 1
df['px_sma50'] = df['Close'] / df['SMA50'] - 1
df['ema_sma']  = df['EMA20'] / df['SMA20'] - 1
df['px_vwap']  = df['Close'] / df['VWAP20'] - 1
df['macd_n']   = (df['MACD'] - df['Signal']) / df['Close']
df['vol_20']   = df.groupby('Ticker')['Return'].transform(lambda s: s.rolling(20).std())
df['rel_vol']  = df['Volume'] / df.groupby('Ticker')['Volume'].transform(lambda s: s.rolling(20).mean())

FEATURES = ['ret_5d','ret_20d','px_sma20','px_sma50','ema_sma',
            'px_vwap','macd_n','vol_20','rel_vol','RSI14']

model_df = df.dropna(subset=FEATURES + ['Label'])
print(f"Rows before: {len(df)}, after dropna: {len(model_df)}")

Rows before: 16302, after dropna: 15470


In [ ]:
!pip install xgboost -q

from xgboost import XGBClassifier
from sklearn.metrics import precision_score

train = model_df[model_df.index < '2024-01-01']
test  = model_df[model_df.index >= '2024-01-01']

model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05)
model.fit(train[FEATURES], train['Label'])
pred = model.predict(test[FEATURES])

print('Win rate in test years :', round(test['Label'].mean(), 3))
print('Model precision        :', round(precision_score(test['Label'], pred), 3))

Win rate in test years : 0.4
Model precision        : 0.429


In [ ]:
always_yes_precision = test['Label'].mean()

mom = (test['ret_20d'] > 0).astype(int)
mom_precision = precision_score(test['Label'], mom)

print('Always-say-yes precision:', round(always_yes_precision, 3))
print('Momentum precision      :', round(mom_precision, 3))

Always-say-yes precision: 0.4
Momentum precision      : 0.393


In [ ]:
proba_train = model.predict_proba(train[FEATURES])[:, 1]
proba_test  = model.predict_proba(test[FEATURES])[:, 1]

def bullish_score(p):
    return round(200 * (proba_train < p).mean() - 100)

test = test.assign(Score=[bullish_score(p) for p in proba_test])

buckets = pd.cut(test['Score'], [-100, -50, 0, 50, 100])
print(test.groupby(buckets)['Label'].agg(win_rate='mean', n='size'))

             win_rate     n
Score                      
(-100, -50]  0.398121  2022
(-50, 0]     0.406589  2307
(0, 50]      0.376640  2363
(50, 100]    0.427119  1770


/tmp/ipykernel_4407/1441823766.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(test.groupby(buckets)['Label'].agg(win_rate='mean', n='size'))


In [ ]:
from xgboost import XGBClassifier

quarters = pd.date_range('2022-01-01', pd.Timestamp.today(), freq='QS')

parts = []
last_model = None

for q_start, q_end in zip(quarters[:-1], quarters[1:]):
    tr = model_df[model_df.index < q_start]
    te = model_df[(model_df.index >= q_start) & (model_df.index < q_end)]

    if len(tr) < 100 or len(te) == 0:
        continue

    m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05)
    m.fit(tr[FEATURES], tr['Label'])
    parts.append(te.assign(Proba=m.predict_proba(te[FEATURES])[:, 1]))
    last_model = m   # keep the most recent model for the Today Sheet later

wf = pd.concat(parts)
print(f"Walk-forward rows: {len(wf)}, quarters used: {len(parts)}")
wf.head()

Walk-forward rows: 14638, quarters used: 18


,Ticker,Open,High,Low,Close,Volume,Label,Year,SMA20,SMA50,...,ret_5d,ret_20d,px_sma20,px_sma50,ema_sma,px_vwap,macd_n,vol_20,rel_vol,Proba
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,TSLA,382.583344,400.356659,378.679993,399.926666,103931400,0.0,2022,340.346500,355.540932,...,0.096751,0.182084,0.175057,0.124840,0.028818,0.181664,0.015926,0.046461,1.484510,0.269348
2022-01-03,JPM,159.860001,162.639999,159.509995,161.699997,13120900,1.0,2022,158.688001,163.227200,...,0.022382,0.021543,0.018981,-0.009356,0.003228,0.018841,0.003724,0.011574,1.155099,0.823690
2022-01-03,AAPL,177.830002,182.880005,177.710007,182.009995,104487900,0.0,2022,175.490000,162.748201,...,0.009316,0.124629,0.037153,0.118353,-0.006765,0.040802,-0.000881,0.019337,0.975410,0.012200
2022-01-03,AVGO,66.632004,67.169998,66.122002,66.332001,22591000,0.0,2022,63.560400,58.423460,...,-0.016254,0.188490,0.043606,0.135366,0.002770,0.052221,0.000199,0.025164,0.806184,0.355132
2022-01-03,META,338.299988,341.079987,337.190002,338.540009,14537900,0.0,2022,334.466000,332.122000,...,-0.022069,0.103311,0.012181,0.019324,0.004782,0.014987,0.002409,0.018085,0.784586,0.134158


In [ ]:
proba_train_full = model_df[model_df.index < '2024-01-01']  # reuse original train split for scaling reference
proba_train_vals = model.predict_proba(proba_train_full[FEATURES])[:, 1]  # from Week 3's model

def bullish_score(p):
    return round(200 * (proba_train_vals < p).mean() - 100)

wf['Score'] = wf['Proba'].apply(bullish_score)
wf[['Ticker', 'Proba', 'Score', 'Label']].head()

,Ticker,Proba,Score,Label
Date,,,,
2022-01-03,TSLA,0.269348,-63,0.0
2022-01-03,JPM,0.823690,99,1.0
2022-01-03,AAPL,0.012200,-100,0.0
2022-01-03,AVGO,0.355132,-4,0.0
2022-01-03,META,0.134158,-98,0.0


In [ ]:
from sklearn.metrics import precision_score

# Same date range for a fair comparison
wf_period = wf
frozen_period = test[test.index.isin(wf.index)]   # frozen Week-3 model's predictions, same dates

wf_precision = precision_score(wf['Label'], wf['Proba'] > 0.5)
frozen_precision = precision_score(frozen_period['Label'], model.predict(frozen_period[FEATURES]))

print('Walk-forward precision   :', round(wf_precision, 3))
print('Frozen Week-3 precision  :', round(frozen_precision, 3))

Walk-forward precision   : 0.39
Frozen Week-3 precision  : 0.429


In [ ]:
backtest_input = wf[['Ticker', 'Close', 'High', 'Low', 'Score']].copy()
backtest_input.head()

# Hand this to Karthika:
# trades, curve = backtest(backtest_input, threshold=0)

,Ticker,Close,High,Low,Score
Date,,,,,
2022-01-03,TSLA,399.926666,400.356659,378.679993,-63
2022-01-03,JPM,161.699997,162.639999,159.509995,99
2022-01-03,AAPL,182.009995,182.880005,177.710007,-100
2022-01-03,AVGO,66.332001,67.169998,66.122002,-4
2022-01-03,META,338.540009,341.079987,337.190002,-98


In [ ]:
# Re-run your data pipeline through yesterday's close, then apply the SAME
# indicator + feature cells (2, 3, 5 above) on this fresh data before scoring.

latest = df.groupby('Ticker').tail(1)   # most recent row per ticker, after all indicators/features are computed

latest = latest.dropna(subset=FEATURES)
latest = latest.assign(Proba=last_model.predict_proba(latest[FEATURES])[:, 1])
latest['Score'] = latest['Proba'].apply(bullish_score)

# Top features driving the model overall
importances = pd.Series(last_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Top features:\n", importances.head(3))

today_sheet = latest[['Ticker', 'Score']].sort_values('Score', ascending=False).reset_index(drop=True)
today_sheet['Date'] = pd.Timestamp.today().date()
today_sheet

Top features:
 vol_20     0.116749
ret_20d    0.107956
RSI14      0.107190
dtype: float32


,Ticker,Score,Date
0,XOM,69,2026-09-08
1,MSFT,47,2026-09-08
2,COST,38,2026-09-08
3,NVDA,36,2026-09-08
4,AMZN,27,2026-09-08
5,GOOGL,27,2026-09-08
6,TSLA,26,2026-09-08
7,AMD,12,2026-09-08
8,AVGO,-6,2026-09-08
9,LLY,-37,2026-09-08


In [ ]:
print(df.index.max())

2026-08-31 00:00:00
